# 🤖 Advanced Model Training with Hyperparameter Tuning
## Cervical Cancer Risk Prediction

**Author:** ML Pipeline  
**Date:** 2026-07-10  
**Purpose:** Train 9 models with hyperparameter tuning using Train/Val/Test split

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import json
import mlflow
import mlflow.sklearn
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import GridSearchCV, RepeatedStratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix
)
import joblib
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / 'processed_data'
MODELS_DIR = PROJECT_ROOT / 'models'
MODELS_DIR.mkdir(exist_ok=True)

MLFLOW_TRACKING_URI = f"sqlite:///{(PROJECT_ROOT / 'notebooks' / 'mlflow.db').as_posix()}"
EXPERIMENT_NAME = "Cervical_Cancer_Model_Training_Leakage_Safe"

print("Libraries imported successfully")
print(f"Project root: {PROJECT_ROOT}")
print(f"MLflow tracking URI: {MLFLOW_TRACKING_URI}")
print(f"Experiment: {EXPERIMENT_NAME}")

## 2. Load Data

In [ ]:
print("=" * 70)
print("LOADING DATA")
print("=" * 70)

X_train = np.load(OUTPUT_DIR / 'X_train.npy')
X_val = np.load(OUTPUT_DIR / 'X_val.npy')
X_test = np.load(OUTPUT_DIR / 'X_test.npy')
y_train = np.load(OUTPUT_DIR / 'y_train.npy')
y_val = np.load(OUTPUT_DIR / 'y_val.npy')
y_test = np.load(OUTPUT_DIR / 'y_test.npy')

with open(OUTPUT_DIR / 'feature_columns.json', 'r') as f:
    feature_names = json.load(f)

print(f"✓ Data loaded successfully:")
print(f"  - Training:      {X_train.shape[0]} samples × {X_train.shape[1]} features")
print(f"  - Validation:    {X_val.shape[0]} samples × {X_val.shape[1]} features")
print(f"  - Test:          {X_test.shape[0]} samples × {X_test.shape[1]} features")

## 3. Setup MLflow

In [ ]:
print("=" * 70)
print("SETTING UP MLFLOW TRACKING")
print("=" * 70)

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

print(f"✓ MLflow tracking URI: {MLFLOW_TRACKING_URI}")
print(f"✓ Experiment: {EXPERIMENT_NAME}")

## 4. Define Models and Hyperparameter Grids

In [ ]:
print("=" * 70)
print("DEFINING REGULARIZED MODELS AND SEARCH GRIDS")
print("=" * 70)

# Preprocessing and SMOTE are fitted inside each CV training fold.
def make_pipeline(estimator):
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('sampler', SMOTE(random_state=42, k_neighbors=3)),
        ('model', estimator),
    ])

models = {
    'Logistic_Regression': make_pipeline(
        LogisticRegression(random_state=42, class_weight='balanced', max_iter=2000)
    ),
    'Logistic_ElasticNet': make_pipeline(
        LogisticRegression(
            random_state=42, solver='saga', penalty='elasticnet', max_iter=3000
        )
    ),
    'Linear_SVM': make_pipeline(
        SVC(kernel='linear', probability=True, class_weight='balanced', random_state=42)
    ),
    'RBF_SVM': make_pipeline(
        SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=42)
    ),
    'Random_Forest': make_pipeline(
        RandomForestClassifier(
            random_state=42, class_weight='balanced_subsample', n_jobs=-1
        )
    ),
    'Gradient_Boosting': make_pipeline(
        GradientBoostingClassifier(random_state=42)
    ),
    'LDA_Shrinkage': make_pipeline(
        LinearDiscriminantAnalysis()
    ),
    'SGD_ElasticNet': make_pipeline(
        SGDClassifier(
            loss='log_loss', penalty='elasticnet', random_state=42,
            max_iter=3000, early_stopping=True, validation_fraction=0.15
        )
    ),
    'Hist_Gradient_Boosting': make_pipeline(
        HistGradientBoostingClassifier(random_state=42)
    ),
}

param_grids = {
    'Logistic_Regression': {
        'model__C': [0.01, 0.1, 1.0, 10.0]
    },
    'Logistic_ElasticNet': {
        'model__C': [0.01, 0.05, 0.1, 0.5],
        'model__l1_ratio': [0.05, 0.15, 0.3]
    },
    'Linear_SVM': {
        'model__C': [0.01, 0.1, 1.0, 10.0]
    },
    'RBF_SVM': {
        'model__C': [0.1, 1.0, 10.0],
        'model__gamma': ['scale', 0.01, 0.1]
    },
    'Random_Forest': {
        'model__n_estimators': [100, 200],
        'model__max_depth': [4, 8, 12],
        'model__min_samples_leaf': [2, 5, 10],
        'model__max_features': ['sqrt', 0.7]
    },
    'Gradient_Boosting': {
        'model__n_estimators': [50, 100],
        'model__learning_rate': [0.03, 0.1],
        'model__max_depth': [1, 2, 3],
        'model__min_samples_leaf': [3, 8]
    },
    'LDA_Shrinkage': [
        {'model__solver': ['lsqr'], 'model__shrinkage': ['auto', None]}
    ],
    'SGD_ElasticNet': {
        'model__alpha': [0.0001, 0.001, 0.01],
        'model__l1_ratio': [0.05, 0.15, 0.3],
        'model__average': [True]
    },
    'Hist_Gradient_Boosting': {
        'model__max_iter': [50, 100],
        'model__learning_rate': [0.03, 0.05],
        'model__max_leaf_nodes': [3, 7],
        'model__min_samples_leaf': [20, 40],
        'model__l2_regularization': [1.0, 10.0]
    }
}

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
print(f"Defined {len(models)} regularized models with fold-local preprocessing")
print("CV: 5 stratified folds x 3 repeats; scoring: F1")

## 5. Train and Tune Models

In [ ]:
print("\n" + "=" * 70)
print("LEAKAGE-SAFE TRAINING AND TUNING")
print("=" * 70)

results = {}

for model_name, model in models.items():
    print(f"\nTraining: {model_name}")
    with mlflow.start_run(run_name=model_name) as run:
        grid_search = GridSearchCV(
            estimator=model,
            param_grid=param_grids[model_name],
            cv=cv,
            scoring='f1',
            n_jobs=-1,
            return_train_score=True,
            verbose=0
        )
        grid_search.fit(X_train, y_train)
        best_model = grid_search.best_estimator_

        predictions = {}
        probabilities = {}
        for split_name, features in [('train', X_train), ('val', X_val), ('test', X_test)]:
            predictions[split_name] = best_model.predict(features)
            probabilities[split_name] = best_model.predict_proba(features)[:, 1]

        metrics = {}
        for split_name, y_true in [('train', y_train), ('val', y_val), ('test', y_test)]:
            y_pred = predictions[split_name]
            y_prob = probabilities[split_name]
            metrics[split_name] = {
                'accuracy': accuracy_score(y_true, y_pred),
                'precision': precision_score(y_true, y_pred, zero_division=0),
                'recall': recall_score(y_true, y_pred, zero_division=0),
                'f1_score': f1_score(y_true, y_pred, zero_division=0),
                'roc_auc': roc_auc_score(y_true, y_prob),
                'pr_auc': average_precision_score(y_true, y_prob),
                'confusion_matrix': confusion_matrix(y_true, y_pred).tolist()
            }

        cv_train = grid_search.cv_results_['mean_train_score'][grid_search.best_index_]
        cv_valid = grid_search.cv_results_['mean_test_score'][grid_search.best_index_]
        cv_std = grid_search.cv_results_['std_test_score'][grid_search.best_index_]

        mlflow.log_params(grid_search.best_params_)
        mlflow.log_param('model_type', model_name)
        mlflow.log_param('cv_strategy', 'RepeatedStratifiedKFold(5x3)')
        mlflow.log_param('resampling', 'SMOTE inside CV pipeline')
        mlflow.log_param('train_size', X_train.shape[0])
        mlflow.log_param('val_size', X_val.shape[0])
        mlflow.log_param('test_size', X_test.shape[0])
        mlflow.log_metric('cv_train_f1_mean', cv_train)
        mlflow.log_metric('cv_validation_f1_mean', cv_valid)
        mlflow.log_metric('cv_validation_f1_std', cv_std)

        for split_name, split_metrics in metrics.items():
            for metric_name, metric_value in split_metrics.items():
                if metric_name != 'confusion_matrix':
                    mlflow.log_metric(f'{split_name}_{metric_name}', round(metric_value, 4))

        mlflow.sklearn.log_model(
            best_model, 'model', serialization_format='pickle'
        )
        joblib.dump(best_model, MODELS_DIR / f'{model_name}_tuned.pkl')

        results[model_name] = {
            'model': best_model,
            'best_params': grid_search.best_params_,
            'best_cv_score': cv_valid,
            'cv_score_std': cv_std,
            'cv_train_score': cv_train,
            'metrics': metrics,
            'run_id': run.info.run_id
        }

        print(f"  Best parameters: {grid_search.best_params_}")
        print(f"  Repeated CV F1: {cv_valid:.4f} +/- {cv_std:.4f}")
        print(f"  Validation F1: {metrics['val']['f1_score']:.4f}")
        print(f"  Test F1: {metrics['test']['f1_score']:.4f}")

## 6. Model Comparison Report

In [ ]:
print("\n" + "=" * 70)
print("MODEL COMPARISON REPORT")
print("=" * 70)

comparison_data = []
for model_name, result in results.items():
    row = {
        'Model': model_name,
        'CV_F1_Mean': result['best_cv_score'],
        'CV_F1_Std': result['cv_score_std'],
        'CV_Train_F1_Mean': result['cv_train_score'],
    }
    for split in ['train', 'val', 'test']:
        for metric in ['accuracy', 'precision', 'recall', 'f1_score', 'roc_auc', 'pr_auc']:
            row[f'{split}_{metric}'] = result['metrics'][split][metric]
    row['train_val_gap'] = row['train_f1_score'] - row['val_f1_score']
    comparison_data.append(row)

df_comparison = pd.DataFrame(comparison_data)
df_comparison = df_comparison.sort_values('CV_F1_Mean', ascending=False).reset_index(drop=True)

print("\nModel rankings by repeated-CV F1 (not test performance):")
display_cols = ['Model', 'CV_F1_Mean', 'CV_F1_Std', 'train_f1_score', 'val_f1_score', 'test_f1_score', 'train_val_gap']
print(df_comparison[display_cols].to_string(index=False))

df_comparison.to_csv(MODELS_DIR / 'model_comparison_advanced.csv', index=False)
# Model selection: highest validation F1 among models whose train-validation
# F1 gap is within 0.05 (ties broken by validation PR-AUC). The registry
# notebook applies the same rule. The test set is never used for selection.
MAX_TRAIN_VAL_GAP = 0.05
candidates = df_comparison[df_comparison['train_val_gap'] <= MAX_TRAIN_VAL_GAP]
if candidates.empty:
    candidates = df_comparison
best_model_name = candidates.sort_values(
    ['val_f1_score', 'val_pr_auc'], ascending=False
).iloc[0]['Model']
print(f"\nSelected model (validation F1, gap <= {MAX_TRAIN_VAL_GAP}): {best_model_name}")
print("The test set is reported here for final comparison only and is not used for selection.")

## 7. Training Summary

In [ ]:
print("\n" + "=" * 70)
print("TRAINING SUMMARY")
print("=" * 70)

print(f"\nModels trained: {len(results)}")
print("Cross-validation: RepeatedStratifiedKFold, 5 folds × 3 repeats")
print("Preprocessing: median imputation, scaling, and SMOTE inside each fold")
print(f"MLflow experiment: {EXPERIMENT_NAME}")

print("\nTop models by repeated-CV F1:")
for rank, (_, row) in enumerate(df_comparison.head(3).iterrows(), 1):
    print(
        f"  {rank}. {row['Model']:22s}: "
        f"CV F1={row['CV_F1_Mean']:.4f} +/- {row['CV_F1_Std']:.4f}, "
        f"Test F1={row['test_f1_score']:.4f}"
    )

print("\nOutput files:")
print("  - models/*_tuned.pkl")
print("  - models/model_comparison_advanced.csv")
print("  - notebooks/mlflow.db")
print("\nTraining completed. Evaluate the selected configuration on the untouched test set once.")